In [1]:
import argparse
import json
import os
from glob import glob

import pandas as pd
from tqdm import tqdm

In [3]:
LABEL_COLUMNS = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Enlarged Cardiomediastinum",
    "Fracture",
    "Lung Lesion",
    "Lung Opacity",
    "Pleural Effusion",
    "Pneumonia",
    "Pneumothorax",
    "Pleural Other",
    "Support Devices",
    "No Finding",
]

FRONTAL_VIEWS = {"PA", "AP"}

In [11]:
def build_caption_from_row(row, include_uncertain=True, drop_no_finding_if_any=True):
    """
    Build a caption as a list of label strings from a NegBio CSV row.

    Values:
      1.0  -> positive
      -1.0 -> uncertain
      0.0  -> explicitly negative
      NaN  -> not mentioned
    """
    positives = []
    uncertain = []
    no_finding = []

    for lbl in LABEL_COLUMNS:
        val = row.get(lbl, float("nan"))
        if pd.isna(val):
            continue

        if lbl == "No Finding":
            if val == 1.0:
                no_finding.append(lbl)
            continue

        if val == 1.0:
            positives.append(lbl)
        elif val == -1.0 and include_uncertain:
            uncertain.append(f"uncertain {lbl}")

    caption_labels = positives + uncertain

    # Handle "No Finding"
    if no_finding:
        if not caption_labels or not drop_no_finding_if_any:
            caption_labels.append("No Finding")

    return caption_labels


def load_metadata(metadata_csv: str):
    """
    Load metadata CSV and return a mapping:
      dicom_id -> ViewPosition (e.g. 'PA', 'AP', 'LATERAL', ...)
    """
    print(f"Loading metadata from: {metadata_csv}")
    mdf = pd.read_csv(metadata_csv)
    if "dicom_id" not in mdf.columns or "ViewPosition" not in mdf.columns:
        raise ValueError("Metadata CSV must contain 'dicom_id' and 'ViewPosition' columns")

    meta = {}
    for _, row in mdf.iterrows():
        did = str(row["dicom_id"])
        view = row["ViewPosition"] if isinstance(row["ViewPosition"], str) else ""
        meta[did] = view
    print(f"Loaded metadata for {len(meta)} dicom_ids")
    return meta


def find_study_dir(base_dir, subject_id, study_id):
    """
    Given integer-like subject_id and study_id, return path to study directory
    in MIMIC-CXR-JPG layout.
    """
    subj_name = f"p{int(subject_id)}"
    study_name = f"s{int(study_id)}"

    patterns = [
        os.path.join(base_dir, "p*", subj_name, study_name),  # p10/p10000032/s50414267
    ]

    for pat in patterns:
        matches = glob(pat)
        if matches:
            return matches[0]

    return None


def collect_pairs_from_negbio(
    base_dir: str,
    negbio_csv: str,
    metadata_csv: str,
    relative_to: str = None,
    max_studies: int = None,
    first_image_only: bool = False,
):
    base_dir = os.path.abspath(base_dir)
    if relative_to is not None:
        relative_to = os.path.abspath(relative_to)

    # Load metadata and NegBio labels
    metadata = load_metadata(metadata_csv)

    print(f"Loading NegBio CSV from: {negbio_csv}")
    df = pd.read_csv(negbio_csv)

    assert "subject_id" in df.columns and "study_id" in df.columns, \
        "NegBio CSV must contain 'subject_id' and 'study_id' columns"

    df = df.sort_values(["subject_id", "study_id"]).reset_index(drop=True)

    if max_studies is not None:
        df = df.head(max_studies)

    pairs = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Studies"):
        subject_id = int(row["subject_id"])
        study_id = int(row["study_id"])

        caption_labels = build_caption_from_row(row)
        if not caption_labels:
            # no useful labels -> skip
            continue

        study_dir = find_study_dir(base_dir, subject_id, study_id)
        if study_dir is None or not os.path.isdir(study_dir):
            # study not present in this subset
            continue

        # All jpg/jpeg images in this study dir
        image_paths = sorted(
            glob(os.path.join(study_dir, "*.jpg"))
            + glob(os.path.join(study_dir, "*.jpeg"))
        )
        if not image_paths:
            continue

        # Filter to frontal views using metadata
        frontal_images = []
        for img_path in image_paths:
            dicom_id = os.path.splitext(os.path.basename(img_path))[0]
            view = metadata.get(dicom_id, "")
            if view in FRONTAL_VIEWS:
                frontal_images.append(img_path)

        if not frontal_images:
            # No frontal image in this study
            continue

        if first_image_only:
            frontal_images = frontal_images[:1]

        for img_path in frontal_images:
            if relative_to is not None:
                img_stored = os.path.relpath(img_path, relative_to)
            else:
                img_stored = img_path

            pairs.append(
                {
                    "image": img_stored,
                    "caption": caption_labels,  # list of label strings
                    # "subject_id": str(subject_id),
                    # "study_id": str(study_id),
                    # "image_id": os.path.basename(img_path),
                }
            )

    return pairs

In [2]:
base_dir = "/home/harsh/Documents/fau/thesis/thesis-codebase/data/mimic_cxr"
negbio_csv = "/home/harsh/Documents/fau/thesis/thesis-codebase/data/mimic_cxr/mimic-cxr-2.0.0-negbio.csv"
metadata_csv = "/home/harsh/Documents/fau/thesis/thesis-codebase/data/mimic_cxr/mimic-cxr-2.0.0-metadata.csv"
output_json = "/home/harsh/Documents/fau/thesis/thesis-codebase/data/mimic_cxr_local.json"

In [13]:
pairs = collect_pairs_from_negbio(
    base_dir=base_dir,
    negbio_csv=negbio_csv,
    metadata_csv=metadata_csv,
    first_image_only=True
)

print(f"Collected {len(pairs)} image-text pairs.")

os.makedirs(os.path.dirname(os.path.abspath(output_json)), exist_ok=True)
with open(output_json, "w", encoding="utf-8") as f:
    json.dump(pairs, f, ensure_ascii=False, indent=2)

print(f"Saved pairs to {output_json}")

Loading metadata from: /home/harsh/Documents/fau/thesis/thesis-codebase/data/mimic_cxr/mimic-cxr-2.0.0-metadata.csv
Loaded metadata for 377110 dicom_ids
Loading NegBio CSV from: /home/harsh/Documents/fau/thesis/thesis-codebase/data/mimic_cxr/mimic-cxr-2.0.0-negbio.csv


Studies: 100%|██████████| 227827/227827 [00:23<00:00, 9732.08it/s] 

Collected 34 image-text pairs.
Saved pairs to /home/harsh/Documents/fau/thesis/thesis-codebase/data/mimic_cxr.json


In [3]:
with open(output_json, "r", encoding="utf-8") as f:
    mimic_cxr_json = json.load(f)

In [4]:
mimic_cxr_json

[{'image': '/home/harsh/Documents/fau/thesis/thesis-codebase/data/mimic_cxr/p10/p10000032/s50414267/02aa804e-bde0afdd-112c0b34-7bc16630-4e384014.jpg',
  'caption': ['No Finding']},
 {'image': '/home/harsh/Documents/fau/thesis/thesis-codebase/data/mimic_cxr/p10/p10000032/s53189527/2a2277a9-b0ded155-c0de8eb9-c124d10e-82c5caab.jpg',
  'caption': ['No Finding']},
 {'image': '/home/harsh/Documents/fau/thesis/thesis-codebase/data/mimic_cxr/p10/p10000032/s53911762/68b5c4b1-227d0485-9cc38c3f-7b84ab51-4b472714.jpg',
  'caption': ['No Finding']},
 {'image': '/home/harsh/Documents/fau/thesis/thesis-codebase/data/mimic_cxr/p10/p10000032/s56699142/ea030e7a-2e3b1346-bc518786-7a8fd698-f673b44c.jpg',
  'caption': ['No Finding']},
 {'image': '/home/harsh/Documents/fau/thesis/thesis-codebase/data/mimic_cxr/p10/p10000764/s57375967/096052b7-d256dc40-453a102b-fa7d01c6-1b22c6b4.jpg',
  'caption': ['Consolidation', 'uncertain Pneumonia']},
 {'image': '/home/harsh/Documents/fau/thesis/thesis-codebase/data/mim

In [7]:
df = pd.DataFrame(mimic_cxr_json)

In [8]:
df.caption.value_counts()

caption
[No Finding]                                             12
[Support Devices]                                         3
[Pleural Effusion]                                        2
[Atelectasis]                                             2
[uncertain Cardiomegaly, uncertain Support Devices]       1
[Edema]                                                   1
[Lung Opacity, uncertain Atelectasis]                     1
[Cardiomegaly, Pleural Effusion]                          1
[Cardiomegaly, Edema, Pleural Effusion]                   1
[Edema, uncertain Consolidation, uncertain Pneumonia]     1
[Consolidation, uncertain Pneumonia]                      1
[Lung Lesion, Pneumonia]                                  1
[uncertain Pneumonia]                                     1
[uncertain Edema]                                         1
[Fracture]                                                1
[Lung Lesion, Lung Opacity]                               1
[Lung Opacity, uncertain Pneumon

In [14]:
with open("/home/harsh/Documents/fau/thesis/thesis-codebase/ALBEF/data/mimic_cxr.json", "r", encoding="utf-8") as f:
    mimic_cxr_json_full = json.load(f)

In [17]:
df_full = pd.DataFrame(mimic_cxr_json_full)
df_full.head()

,image,caption
0,/home/woody/iwi5/iwi5362h/data/mimic_cxr/extra...,[No Finding]
1,/home/woody/iwi5/iwi5362h/data/mimic_cxr/extra...,[No Finding]
2,/home/woody/iwi5/iwi5362h/data/mimic_cxr/extra...,[No Finding]
3,/home/woody/iwi5/iwi5362h/data/mimic_cxr/extra...,[No Finding]
4,/home/woody/iwi5/iwi5362h/data/mimic_cxr/extra...,"[Consolidation, uncertain Pneumonia]"


In [18]:
df_full.caption.value_counts()

caption
[No Finding]                                                                                                                        65388
[Support Devices]                                                                                                                   11126
[Cardiomegaly]                                                                                                                       5026
[Atelectasis]                                                                                                                        4177
[Pleural Effusion]                                                                                                                   3400
                                                                                                                                    ...  
[Consolidation, Lung Opacity, uncertain Atelectasis, uncertain Edema, uncertain Pneumonia]                                              1
[Atelectasis, Consolidatio

In [19]:
mimic_cxr_debug = mimic_cxr_json_full[:200]  # creating a smaller version for debugging on A100 in the cluster

In [24]:
len(mimic_cxr_debug)

200

In [25]:
with open("/home/harsh/Documents/fau/thesis/thesis-codebase/ALBEF/data/mimic_cxr_debug.json", "w", encoding="utf-8") as f:
    json.dump(mimic_cxr_debug, f, ensure_ascii=False, indent=2)